<a href="https://colab.research.google.com/github/cleophasmashiri/ai-jupter-notebooks/blob/main/gpt_dev_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building a GPT — Tutorial Edition

This is the fill-in-the-blank companion to [`gpt_dev.ipynb`](https://colab.research.google.com/github/cleophasmashiri/ai-jupter-notebooks/blob/main/gpt_dev.ipynb) (the fully-annotated Zero To Hero GPT notebook). Every code cell there has been turned into an exercise here: the explanations stay, but the implementation lines are blanked out with `...` for you to fill in, and every exercise has a `pytest` test cell right after it that tells you whether you got it right.

If you get stuck on *why* something works, the fully-worked `gpt_dev.ipynb` has the answer and a deep explanation next to it — this notebook is for testing whether you actually understood it well enough to write it yourself.

### How this notebook works

1. Cells come in groups of three: **instructions** (markdown) → **your code** (with `...` blanks) → **tests** (a `%%ipytest` cell).
2. Fill in the blanks in the code cell, run it, then run the test cell immediately after.
3. A green pytest summary (`X passed`) means you're done with that exercise — move to the next one.
4. A red `FAILED` means something's off — read the assertion message, it's written to tell you specifically what's wrong.
5. Work **top to bottom**. Later exercises depend on variables defined by earlier ones (`vocab_size`, `encode`, `train_data`, `get_batch`, ...), exactly like the original notebook.
6. Plain code cells with no blanks and no test (data downloads, print statements, given scaffolding) are provided as-is — just run them.

### Roadmap

We'll build a GPT from the ground up, in the same order you'd derive it if you were inventing it yourself:

1. **Data → tokens**: turn raw text into integers a neural net can consume.
2. **A trivial baseline**: a bigram model (predict the next character from just the current one) to get the training loop and loss working end-to-end.
3. **The attention trick**: derive self-attention starting from "average the past tokens" and building up to full scaled dot-product attention.
4. **Assemble the Transformer**: stack multi-head attention + feed-forward blocks with residual connections and LayerNorm — this is the actual GPT architecture (minus scale).
5. **Train and sample**: train the full model on Shakespeare and generate new text from it.

Everything here is character-level (not word/subword) to keep the vocabulary tiny and the code simple — the ideas transfer directly to a real GPT-2/3-style BPE tokenizer.

### Glossary (reference while reading)

| Term | Meaning |
|---|---|
| **Token** | An integer id standing in for a chunk of text (here: one character). Everything the model does operates on tokens, never raw text. |
| **Vocabulary / `vocab_size`** | The number of distinct tokens the model can emit — the width of its final output layer. |
| **Embedding** | A learned lookup table mapping a token id to a dense vector (`nn.Embedding`). Turns a discrete id into something a neural net can do arithmetic on. |
| **Logits** | Raw, unnormalized scores output by the model for each possible next token — turned into probabilities by `softmax`. |
| **`block_size` / context length** | The maximum number of past tokens the model can condition on when predicting the next one. |
| **Autoregressive** | Generating a sequence one token at a time, each new token conditioned on all previous ones. |
| **Causal mask** | The mechanism (here, `tril` + `-inf` fill) that prevents a position from attending to future positions — required so training never "cheats" by seeing the answer. |
| **Q, K, V (Query, Key, Value)** | Three learned projections of a token's embedding used by attention: Q = "what I'm looking for," K = "what I offer to be matched against," V = "what I actually send if attended to." |
| **Attention head** | One instance of the Q/K/V mechanism, producing one particular way of mixing information across tokens. |
| **Multi-head attention** | Several attention heads run in parallel (on the same input) and concatenated, so the model can mix context in several different ways at once. |
| **Residual / skip connection** | Adding a layer's output back onto its input (`x = x + f(x)`) instead of replacing it — keeps gradients flowing through deep stacks. |
| **LayerNorm** | Normalizes each token's feature vector to zero mean / unit variance before a sublayer, stabilizing training. |
| **Transformer block** | One layer of self-attention + feed-forward (each wrapped in residual + LayerNorm) — GPT is a stack of these. |
| **Cross-entropy loss** | The training objective: negative log-probability the model assigned to the actual next token. Lower is better; `-ln(1/vocab_size)` is the "random guessing" baseline. |
| **`n_embd`, `n_head`, `n_layer`** | Core size hyperparameters: embedding width, number of attention heads per block, and number of stacked blocks. |

Refer back to this table any time a term feels unfamiliar further down — it isn't meant to be memorized up front.

In [ ]:
!pip install -q ipytest
import ipytest
ipytest.autoconfig()

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:   # open the downloaded file, decoding it as UTF-8
    text = f.read()                                    # read the whole file into one big Python string

In [ ]:
print("length of dataset in characters: ", len(text))   # len() on a string counts characters

In [ ]:
# let's look at the first 1000 characters
print(text[:1000])   # slice the first 1000 characters to eyeball the raw text

**Tokenization, in the simplest possible form.** A neural net can't consume raw text — it needs integers (and eventually vectors). Here we use the crudest possible tokenizer: one token per *character*. `vocab_size` (65) is the number of distinct symbols the model will ever need to predict over — it directly sets the width of the model's output layer (a probability distribution over 65 classes at every position).

Real GPT models use *subword* tokenizers (e.g. GPT-2's BPE, ~50k tokens) instead of characters — that's a tradeoff between sequence length and vocabulary size: fewer, richer tokens mean shorter sequences per unit of text (cheaper, since attention is O(T²)), at the cost of a bigger, sparser output layer and a separate tokenizer-training step. Character-level keeps this notebook self-contained.

### Exercise 1 — Build the vocabulary

Compute the sorted list of every unique character that appears in `text`, and store how many there are.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
chars = ...       # sorted list of unique characters in `text`
vocab_size = ...  # how many unique characters there are

print(''.join(chars))
print(vocab_size)

In [ ]:
%%ipytest -qq

def test_chars_is_sorted_unique():
    assert chars == sorted(set(text)), "chars should be the sorted list of unique characters in `text`"

def test_vocab_size():
    assert vocab_size == 65, f"expected 65 unique characters in tiny shakespeare, got {vocab_size}"
    assert vocab_size == len(chars)

`stoi`/`itos` are just the two halves of a bijection between characters and integers in `[0, vocab_size)`. `encode`/`decode` are the tokenizer's `.encode()`/`.decode()` in miniature — this *is* what `tiktoken` or a HuggingFace tokenizer does, just without merge rules. Keep this mental model: everywhere below, "token" means "integer id from this table," and the model never sees a character — only these ids (and later, embeddings of these ids).

### Exercise 2 — Build the character tokenizer

Build `stoi`/`itos` lookup dicts mapping characters to integer ids and back, then `encode`/`decode` functions built on top of them.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
stoi = ...    # dict: character -> its index in `chars`
itos = ...    # dict: index -> character (the inverse of stoi)
encode = ...  # function/lambda: str -> list[int]
decode = ...  # function/lambda: list[int] -> str

print(encode("hii there"))
print(decode(encode("hii there")))

In [ ]:
%%ipytest -qq

def test_stoi_itos_are_inverses():
    for i, ch in enumerate(chars):
        assert stoi[ch] == i
        assert itos[i] == ch

def test_encode_decode_roundtrip():
    assert decode(encode("hello world")) == "hello world"
    assert encode("a") == [stoi['a']]
    assert len(encode(text[:100])) == 100

The entire dataset becomes one long 1D tensor of integer ids, `dtype=torch.long` (i.e. int64) because `nn.Embedding` and `F.cross_entropy` both require integer *index* tensors, not floats. Nothing about the text's structure (sentences, characters, punctuation) survives here except order — the model has to (re)discover Shakespeare's structure purely from this sequence of integers.

# Understanding `torch.tensor()` and `dtype`

`torch.tensor()` builds a new tensor from Python data (a list, a nested list, a number...). The `dtype` argument controls what kind of numbers it holds — this matters once you start indexing into embedding tables or computing losses.

### Syntax
```python
torch.tensor(data, dtype=None)
```
**Parameters:**
- `data`: the Python data to wrap (list, nested list, scalar, etc.)
- `dtype` (optional): the tensor's storage type, e.g. `torch.long` (64-bit int), `torch.float32` (default for floats). If omitted, PyTorch infers it from `data`.

### Example 1 — basic usage
```python
torch.tensor([1, 2, 3])
```
```
tensor([1, 2, 3])
```
Inferred dtype here is `torch.int64` (a.k.a. `torch.long`), since all the inputs are plain Python ints.

### Example 2 — forcing a dtype
```python
torch.tensor([1, 2, 3], dtype=torch.float32)
```
```
tensor([1., 2., 3.])
```
Same values, now stored as 32-bit floats — notice the trailing `.` on each number.

### In this notebook
```python
data = torch.tensor(encode(text), dtype=torch.long)
```
`encode(text)` returns a plain Python `list[int]` (one integer per character). Wrapping it in `torch.tensor(..., dtype=torch.long)` turns that list into a single 1D tensor of 64-bit integers — `torch.long` specifically, because two things downstream require integer *indices*, not floats: `nn.Embedding` (looks up rows by integer index) and `F.cross_entropy` (expects integer class labels).

### Summary
| Call | Result |
|---|---|
| `torch.tensor([1,2,3])` | int64 tensor, dtype inferred |
| `torch.tensor([1,2,3], dtype=torch.float32)` | same values as 32-bit floats |
| `torch.tensor(encode(text), dtype=torch.long)` | the entire dataset as one 1D int64 tensor of token ids |

### Exercise 3 — Encode the dataset into a tensor

Encode the *entire* `text` and wrap it in a `torch.long` tensor.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
import torch
data = ...  # the whole dataset, encoded, as a torch.long tensor

print(data.shape, data.dtype)
print(data[:1000])

In [ ]:
%%ipytest -qq

def test_data_dtype_and_shape():
    assert data.dtype == torch.long
    assert data.shape[0] == len(text)

def test_data_matches_encode():
    assert data[:5].tolist() == encode(text[:5])

Note this is a **sequential** split (first 90% / last 10%), not a random shuffle. Two reasons: (1) it prevents leakage — random windows would let validation chunks overlap almost entirely with training chunks just a few characters away; (2) it mimics the real evaluation setting for a language model — "trained on the past, tested on what comes after" — rather than an i.i.d. assumption that doesn't really hold for text.

### Exercise 4 — Train / validation split

Split `data` sequentially: the first 90% is `train_data`, the rest is `val_data`. No shuffling.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
n = ...           # index marking the 90% cutoff
train_data = ...  # data[:n]
val_data = ...    # data[n:]

In [ ]:
%%ipytest -qq

def test_split_sizes():
    assert n == int(0.9 * len(data))
    assert len(train_data) + len(val_data) == len(data)

def test_split_is_sequential_not_shuffled():
    assert torch.equal(train_data, data[:n])
    assert torch.equal(val_data, data[n:])

`block_size` (aka **context length** / `n_ctx` in GPT papers) is the maximum number of previous tokens the model is allowed to look at when predicting the next one. It's a hyperparameter you choose, not something inherent to the data — increasing it lets the model condition on more history, at the cost of attention's O(block_size²) compute and memory. GPT-3 uses 2048, modern long-context models use tens of thousands to millions; here we use 8 (later bumped to 32) purely so the toy examples below print legibly.

### Exercise 5 — block_size and a single chunk

Set `block_size = 8` and slice out `block_size + 1` tokens from `train_data` (one more than `block_size`, since that one extra token is needed to form the last target).

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
block_size = ...  # max context length
chunk = ...       # train_data[:block_size+1]
chunk

In [ ]:
%%ipytest -qq

def test_block_size_value():
    assert block_size == 8

def test_chunk_length():
    assert chunk.shape[0] == block_size + 1
    assert torch.equal(chunk, train_data[:block_size + 1])

This loop is the key insight behind training-example construction: a single chunk of `block_size+1` tokens yields `block_size` *separate* training examples, one per prefix length (context of length 1, 2, 3, ... up to `block_size`). This is what makes the model useful at inference time with any context length from 1 up to `block_size` — it was explicitly trained on all of those lengths, not just the full one. It's also why the target `y` is just `x` shifted by one position: next-token prediction is the entire task, and every position in a sequence supplies its own training signal for free.

### Exercise 6 — Context → target pairs

Build `x` (the first `block_size` tokens) and `y` (the same tokens shifted by one position — i.e. `x`'s next-token labels).

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
x = ...  # train_data[:block_size]
y = ...  # train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

In [ ]:
%%ipytest -qq

def test_x_y_shapes_and_values():
    assert torch.equal(x, train_data[:block_size])
    assert torch.equal(y, train_data[1:block_size + 1])

def test_y_is_shifted_x():
    for t in range(block_size):
        assert y[t].item() == train_data[t + 1].item()

# Understanding `torch.randint()`

`torch.randint()` generates random integers within a specified range. It is commonly used in PyTorch to randomly sample indices, labels, or batches of data — exactly how `get_batch` (next cell) picks where each training chunk starts.

### Syntax

```python
torch.randint(low, high, size)
# or
torch.randint(high, size)
```

**Parameters:**
- `low` (optional): lowest integer, inclusive. Defaults to 0.
- `high`: highest integer, exclusive.
- `size`: shape of the output tensor.

### Example 1 — basic usage

```python
import torch
torch.randint(10, (5,))
```
```
tensor([3, 8, 1, 5, 7])
```
Generates 5 random integers, each in `[0, 10)` — 10 itself is excluded. Think of it as randomly picking five numbers from `0 1 2 3 4 5 6 7 8 9`.

### Example 2 — specify a lower bound

```python
torch.randint(5, 10, (6,))
```
```
tensor([6, 8, 5, 9, 7, 6])
```
Values come from `5 6 7 8 9`. `5` is included, `10` is excluded — the interval is `[5, 10)`, where `[` means inclusive and `)` means exclusive.

### Example 3 — a 2D tensor

```python
torch.randint(0, 100, (3, 4))
```
```
tensor([[12, 45, 98,  2],
        [77,  5, 16, 63],
        [44, 29, 81, 70]])
```
`size=(3, 4)` produces 3 rows and 4 columns of independent random draws.

### The line from `get_batch`

```python
batch_size = 4
block_size = 8
ix = torch.randint(len(data) - block_size, (batch_size,))
```

Suppose (for illustration) `len(data) = 1000`. Then `len(data) - block_size = 1000 - 8 = 992`, so PyTorch executes:

```python
ix = torch.randint(992, (4,))   # equivalent to torch.randint(0, 992, (4,))
```
```
tensor([101, 540, 220, 905])
```
These four numbers are random **starting positions** inside the dataset. (In the real notebook, `len(data)` is `train_data`'''s actual length, ~1,003,854 — the arithmetic is identical, just with a bigger number.)

### Why subtract `block_size`?

Later in the code: `data[i : i + block_size]` — each training example needs `block_size` consecutive tokens starting at `i`. If `i = 998` (with `len(data)=1000`), then `998 + 8 = 1006`, which runs past the end of the dataset. Capping `i` at `991` gives `991 + 8 = 999` — the last valid index — so the slice always stays inside the dataset.

### Visual example

Dataset indices: `0 1 2 3 4 5 6 7 8 9 ... 999`

Suppose `ix = tensor([5, 20, 100, 300])`. These become the starting positions:

```
Example 1: 5   6   7   8   9   10  11  12
Example 2: 20  21  22  23  24  25  26  27
Example 3: 100 101 102 103 104 105 106 107
Example 4: 300 301 302 303 304 305 306 307
```
Each sequence contains exactly 8 tokens.

### Why is the output shape `(batch_size,)`?

```python
batch_size = 4
torch.randint(992, (4,))
```
```
tensor([15, 201, 700, 512])
```
Each integer is one random starting index, used here:

```python
x = torch.stack([data[i : i + block_size] for i in ix])
```
which is equivalent to stacking:
```
data[15  : 23]
data[201 : 209]
data[700 : 708]
data[512 : 520]
```
After stacking, `x` has shape `(4, 8)` — 4 sequences (batch size) of 8 tokens each (block size).

### Summary

| Call | Result |
|---|---|
| `torch.randint(10, (5,))` | 5 random integers from 0 to 9 |
| `torch.randint(5, 10, (5,))` | 5 random integers from 5 to 9 |
| `torch.randint(100, (2, 3))` | a 2×3 matrix of random integers from 0 to 99 |
| `torch.randint(len(data) - block_size, (batch_size,))` | random starting positions for training sequences that fit completely inside the dataset |

In language model training, `torch.randint()` is used to randomly sample starting positions so every training step sees a different slice of the dataset. This randomness helps the model learn more effectively and prevents it from always training on the data in the same order.

# Understanding `torch.stack()`

`torch.stack()` takes a list of tensors that all have the *same shape* and combines them along a brand-new dimension. It's how a Python list of individual examples becomes one batched tensor.

### Syntax
```python
torch.stack(tensors, dim=0)
```
**Parameters:**
- `tensors`: a sequence (e.g. list) of tensors, all with identical shape.
- `dim` (optional): which axis to insert as the new dimension. Defaults to `0` (a new leading dimension).

### Example 1 — stacking 1D tensors into a 2D tensor
```python
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])
torch.stack([a, b])
```
```
tensor([[1, 2, 3],
        [4, 5, 6]])
```
Two shape-`(3,)` tensors become one shape-`(2, 3)` tensor — a new dimension (size 2, one slot per input tensor) was added at position 0.

### Example 2 — a list comprehension of slices
```python
data = torch.arange(20)
ix = [0, 5, 10]
torch.stack([data[i:i+4] for i in ix])
```
```
tensor([[ 0,  1,  2,  3],
        [ 5,  6,  7,  8],
        [10, 11, 12, 13]])
```
Three independent 4-element slices, stacked into one `(3, 4)` tensor — this is exactly the pattern `get_batch` uses.

### In this notebook
```python
x = torch.stack([data[i:i+block_size] for i in ix])
```
For each of the `batch_size` random starts in `ix`, the list comprehension slices out one `(block_size,)` chunk of token ids. `torch.stack` then combines those `batch_size` separate 1D tensors into a single 2D tensor of shape `(batch_size, block_size)` — the `x` that gets fed to the model.

### Summary
| Call | Result |
|---|---|
| `torch.stack([a, b])` where `a, b` are shape `(3,)` | shape `(2, 3)` |
| `torch.stack([data[i:i+block_size] for i in ix])` | shape `(batch_size, block_size)` — a batch of sequences |

### Exercise 7 — get_batch — random minibatches

Implement `get_batch`: pick `batch_size` random starting offsets into the chosen split (making sure a full `block_size`-length window plus one extra target token always fits), then stack the corresponding input and target windows into two `(batch_size, block_size)` tensors.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = ...  # batch_size random starting offsets, valid for a block_size window + 1 target token
    x = ...   # stack of `batch_size` chunks: data[i : i+block_size]
    y = ...   # same offsets, shifted by 1: data[i+1 : i+block_size+1]
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

In [ ]:
%%ipytest -qq

def test_get_batch_shapes():
    x, y = get_batch('train')
    assert x.shape == (batch_size, block_size)
    assert y.shape == (batch_size, block_size)

def test_get_batch_y_is_shifted_x():
    x, y = get_batch('train')
    assert torch.equal(y[:, :-1], x[:, 1:]), "y should be x shifted by one position (same starting offsets)"

def test_get_batch_val_split_works():
    x, y = get_batch('val')
    assert x.shape == (batch_size, block_size)
    assert torch.equal(y[:, :-1], x[:, 1:])

### `get_batch`, line by line

This function is the actual data loader — every training step calls it once to get a fresh minibatch. Worth understanding exactly, since bugs in a from-scratch training loop tend to live here.

```python
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y
```

- **`data = train_data if split == 'train' else val_data`** — pick which 1D token tensor to sample from, depending on whether the caller wants a training or validation batch. `train_data`/`val_data` are the same sequential split from earlier — no shuffling happens here either.
- **`ix = torch.randint(len(data) - block_size, (batch_size,))`** — draw `batch_size` random integers, each uniform in `[0, len(data) - block_size)`. This is *where each sampled chunk starts*. The upper bound is deliberately `len(data) - block_size`, not `len(data)`: the largest chunk we slice is `data[i : i+block_size]`, and its last target position needs `data[i+block_size]` to exist (see `y` below) — so `i` must never get closer than `block_size` tokens to the end of `data`, or the slice would silently come up short. `(batch_size,)` is the *shape* of the output — one random start per example in the batch.
- **`x = torch.stack([data[i:i+block_size] for i in ix])`** — for every sampled start `i`, slice out `block_size` consecutive tokens. Each slice is a 1D tensor of shape `(block_size,)`; the list comprehension produces `batch_size` of them; `torch.stack` stacks them along a new leading dimension, giving one 2D tensor of shape `(batch_size, block_size)` — this is `x`, i.e. `B` sequences of length `T`.
- **`y = torch.stack([data[i+1:i+block_size+1] for i in ix])`** — the crucial trick: **reuse the exact same `ix`**, but slice starting one token later. Since `x[b] = data[i : i+block_size]` and `y[b] = data[i+1 : i+block_size+1]`, `y[b]` is *literally* `x[b]` shifted left by one position — so `y[b, t]` is always the token that comes right after `x[b, t]` in the original text. That is the entire label-construction step for next-token prediction: no separate labels are ever created, they are just an offset view of the same data.
- **`return x, y`** — both shape `(batch_size, block_size)`, ready to feed straight into the model'''s `forward(x, y)`.

**Things that are easy to miss:**
- Sampling is done **with replacement, every call** — there is no concept of an "epoch" here (going through all the data once, without repeats). Each training step draws a fresh set of random windows, some of which may overlap with windows used in earlier or even the *same* step. Over `max_iters` steps this still covers the data thoroughly — this is standard for a from-scratch demo like this; production training loops often add proper epoch-based shuffling instead.
- **Every row in `y` reuses the same starting indices as `x`** (`ix`) — that is what keeps input/target pairs aligned. A common bug when modifying this code is drawing separate random indices for `x` and `y`, which silently produces garbage (uncorrelated) training pairs.
- The bound `len(data) - block_size` (not `len(data) - block_size - 1`) works because `torch.randint(high, ...)` samples from `[0, high)` — `high` itself is excluded — so the largest possible `i` is already `len(data) - block_size - 1`, exactly what is needed so `i + block_size` never runs past the last index of `data`.

The version inside the full model further down is identical except for one added line, `x, y = x.to(device)`, which moves the batch onto the GPU if one is available.

In [ ]:
print(xb) # our input to the transformer

Stacking many `(context, target)` pairs into a batch (`B` = batch dimension) is purely a GPU-efficiency trick — the examples are otherwise independent and never interact with each other (that stays true throughout this notebook: attention lets tokens within a sequence talk to each other, but never across batch elements). So every tensor from here on has shape `(B, T, ...)`: `B` = batch, `T` = time/sequence position (≤ `block_size`), and later `C` = channel/embedding dimension. Keep track of these three axes — nearly every bug in transformer code is a shape mismatch between them.

### Checkpoint: data & tokenization

Before moving on, make sure you can answer these without scrolling back — try each one, then click to reveal the answer.

1. Why is `vocab_size` exactly the width of the model's final output layer?

<details><summary>Show answer</summary>

At every position the model's job is to output a probability distribution over "what token comes next," and there are exactly `vocab_size` possible next tokens. The final layer (the embedding table itself in the toy bigram model, or `lm_head` in the full model) has to emit one logit per possible token — so its output width must equal `vocab_size`.

</details>

2. Why is the train/val split done sequentially instead of with a random shuffle?

<details><summary>Show answer</summary>

Two reasons: (1) it prevents leakage — random windows would let validation chunks overlap almost entirely with training chunks just a few characters away; (2) it mimics the real evaluation setting for a language model ("trained on the past, tested on what comes after") rather than assuming the data is i.i.d., which text isn't.

</details>

3. If `block_size = 8`, how many distinct `(context, target)` training examples does a single 9-token chunk produce, and why?

<details><summary>Show answer</summary>

8 examples. A chunk of `block_size + 1` tokens gives one `(context, target)` pair for every prefix length from 1 up to `block_size` — context of length 1→2, length 2→3, ..., length 8→9. That's what lets the trained model handle any context length from 1 to `block_size` at inference time, not just the full one.

</details>

4. Why does every tensor from here on have a leading batch dimension `B`, even though batch elements never interact with each other?

<details><summary>Show answer</summary>

Purely a GPU-efficiency trick. Stacking many independent `(context, target)` pairs into one tensor lets a single vectorized forward/backward pass process all of them at once, instead of looping one example at a time. The examples never interact with each other — that stays true even once attention is introduced, since attention mixes information across the `T` (time) axis but never across the `B` (batch) axis.

</details>

If any of these aren't automatic, revisit the corresponding note above before continuing — everything past this point builds on it.

# Understanding `nn.Embedding()`

`nn.Embedding` is a lookup table: give it an integer index, it gives you back a learned vector. It's the standard way to turn discrete tokens (ids) into continuous vectors a neural net can compute with.

### Syntax
```python
nn.Embedding(num_embeddings, embedding_dim)
```
**Parameters:**
- `num_embeddings`: how many distinct rows the table has (e.g. `vocab_size`).
- `embedding_dim`: how wide each row's vector is.

Internally it just holds a `(num_embeddings, embedding_dim)` weight matrix, randomly initialized and trained like any other parameter.

### Example 1 — basic lookup
```python
emb = nn.Embedding(10, 3)   # 10 possible ids, each mapped to a 3-dim vector
emb(torch.tensor([1, 4]))
```
```
tensor([[ 0.13, -0.92,  0.47],
        [-0.55,  0.31,  1.02]])
```
Shape `(2, 3)` — one row of the table per input id (values differ every run, since the table is randomly initialized).

### Example 2 — batched input
```python
emb(torch.tensor([[1, 4], [0, 9]])).shape   # (2,2) in -> (2,2,3) out
```
`nn.Embedding` accepts input of *any* shape and just appends `embedding_dim` as a new last dimension — every integer in the input gets replaced by its corresponding row.

### In this notebook
```python
self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)   # toy bigram model
...
self.token_embedding_table = nn.Embedding(vocab_size, n_embd)        # full model
self.position_embedding_table = nn.Embedding(block_size, n_embd)
```
In the toy bigram model, the table is `(vocab_size, vocab_size)` — row `i` directly *is* that token's logits for "what comes next," so `logits = self.token_embedding_table(idx)` needs no further layers. In the full model, `token_embedding_table` maps each of the `vocab_size` possible ids to an `n_embd`-dim vector, and a *separate* `position_embedding_table` maps each of the `block_size` possible positions to its own `n_embd`-dim vector — the two get added together so every token's representation encodes both "what" and "where".

### Summary
| Call | Shape produced |
|---|---|
| `nn.Embedding(vocab_size, vocab_size)(idx)` where `idx` is `(B,T)` | `(B,T,vocab_size)` — used directly as logits |
| `nn.Embedding(vocab_size, n_embd)(idx)` where `idx` is `(B,T)` | `(B,T,n_embd)` — a dense representation, not logits |

# Understanding `tensor.view()`

`.view()` reinterprets a tensor's existing data as a different shape, without copying or moving any values — it just changes how the same flat block of numbers is "sliced up" into dimensions. The total number of elements must stay the same.

### Syntax
```python
tensor.view(*shape)
```
Pass the target shape, either as separate arguments or a tuple. One dimension may be `-1`, meaning "infer this size from the others."

### Example 1 — flattening into one dimension
```python
x = torch.arange(12).view(3, 4)
x.view(12)          # or x.view(-1)
```
```
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
```
Same 12 numbers, same memory, now seen as one long row instead of a `3×4` grid.

### Example 2 — merging leading dimensions
```python
x = torch.zeros(4, 8, 65)     # e.g. (B, T, vocab_size)
x.view(4*8, 65).shape          # torch.Size([32, 65])
```
The first two axes collapse into one axis of size `4*8=32`; the last axis (65) is untouched.

### In this notebook
```python
B, T, C = logits.shape
logits = logits.view(B*T, C)
targets = targets.view(B*T)
loss = F.cross_entropy(logits, targets)
```
`logits` starts as `(B, T, vocab_size)` — a prediction for every position in every sequence in the batch. `F.cross_entropy` doesn't know or care about the batch/time distinction; it just wants "one row of scores per example." So `.view(B*T, C)` merges `B` and `T` into a single axis of `B*T` independent examples, and `targets.view(B*T)` does the matching flatten on the label side.

### Summary
| Before | `.view(...)` | After |
|---|---|---|
| `(B, T, C)` | `.view(B*T, C)` | `(B*T, C)` — one row per (batch, position) pair |
| `(B, T)` | `.view(B*T)` | `(B*T,)` |

# Understanding `F.cross_entropy()`

`F.cross_entropy` is the standard loss for classification: given raw, unnormalized scores (**logits**) and the correct class index for each example, it computes how "surprised" the model was by the truth, on average.

### Syntax
```python
F.cross_entropy(input, target)
```
**Parameters:**
- `input`: shape `(N, C)` — `N` examples, `C` raw scores (logits) each. **Not** pre-softmaxed — this function applies softmax internally.
- `target`: shape `(N,)` — the correct class index (an integer in `[0, C)`) for each of the `N` examples.

Computes, and averages over all `N` examples:
$$\text{loss} = -\log \frac{e^{\text{input}[i,\,target[i]]}}{\sum_j e^{\text{input}[i,j]}}$$
— i.e. the negative log of the softmax probability the model assigned to the *correct* class.

### Example 1 — a single example
```python
logits = torch.tensor([[2.0, 0.5, 0.1]])   # 1 example, 3 classes
target = torch.tensor([0])                  # correct class is index 0
F.cross_entropy(logits, target)
```
```
tensor(0.328)
```
The model gave the correct class (index 0) the highest logit, so the loss is fairly low. If `target` were `2` instead (the *lowest*-scored class), the loss would be much higher.

### Example 2 — a uniform / random baseline
```python
logits = torch.zeros(1, 65)       # every class equally likely
F.cross_entropy(logits, torch.tensor([0]))
```
```
tensor(4.174)
```
This is `-log(1/65)` — the loss from guessing uniformly at random over 65 classes. Any trained model should end up well below this.

### In this notebook
```python
logits = logits.view(B*T, C)
targets = targets.view(B*T)
loss = F.cross_entropy(logits, targets)
```
`C = vocab_size = 65`, `N = B*T` (every position, in every sequence, in the batch, treated as one independent classification example: "given the context up to here, which of the 65 characters comes next?"). The printed losses in this notebook (~4.87 untrained, ~1.66 after training the full model) are directly comparable to the `4.17` random baseline above.

### Summary
| Situation | Loss (over `vocab_size=65`) |
|---|---|
| Uniform random guessing | `-log(1/65) ≈ 4.17` |
| Untrained bigram model | `≈ 4.87` |
| Fully trained model (5000 steps) | `≈ 1.66` (train) / `1.82` (val) |

# Understanding `F.softmax()`

`softmax` turns any vector of raw scores into a valid probability distribution — every entry becomes positive, and the entries sum to exactly 1. Larger input scores become larger (but not linearly larger) probabilities.

### Syntax
```python
F.softmax(input, dim)
```
**Parameters:**
- `input`: a tensor of raw scores (logits).
- `dim`: which axis to normalize over (i.e. which axis's values should sum to 1). Almost always the *last* axis, `dim=-1`, for "one distribution per row."

Formula, applied along `dim`:
$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

### Example 1 — basic usage
```python
F.softmax(torch.tensor([1.0, 2.0, 3.0]), dim=-1)
```
```
tensor([0.0900, 0.2447, 0.6652])
```
Sums to 1. The largest input (`3.0`) gets the largest probability, but the relationship is exponential, not proportional.

### Example 2 — row-wise on a 2D tensor
```python
F.softmax(torch.tensor([[1.0, 2.0], [0.0, 0.0]]), dim=-1)
```
```
tensor([[0.2689, 0.7311],
        [0.5000, 0.5000]])
```
`dim=-1` normalizes each *row* independently — row 2's equal logits give a uniform 50/50 split.

### Example 3 — masking with `-inf` before softmax
```python
F.softmax(torch.tensor([1.0, 2.0, float('-inf')]), dim=-1)
```
```
tensor([0.2689, 0.7311, 0.0000])
```
`-inf` becomes exactly `0` probability after softmax — this is exactly how the causal mask blocks attention to future positions.

### In this notebook
Two distinct uses: (1) in `generate`, `probs = F.softmax(logits[:, -1, :], dim=-1)` turns the model's raw next-character scores into sampling probabilities; (2) in every attention `Head`, `wei = F.softmax(wei, dim=-1)` turns masked attention scores into per-row attention weights that sum to 1 across the allowed (non-future) positions.

### Summary
| Input | `dim=-1` output |
|---|---|
| `[1, 2, 3]` | `[0.09, 0.24, 0.67]` — favors the largest logit, smoothly |
| `[x, x, -inf]` | last entry's probability forced to exactly `0` |

# Understanding `torch.multinomial()`

`torch.multinomial` draws random samples from a categorical distribution, weighted by a vector of probabilities (or unnormalized weights). It's how this notebook turns "the model's predicted probabilities" into an actual, single, sampled next token.

### Syntax
```python
torch.multinomial(input, num_samples)
```
**Parameters:**
- `input`: a 1D or 2D tensor of non-negative weights (doesn't strictly need to sum to 1 — `softmax` output already does).
- `num_samples`: how many indices to draw.

The output is *indices into `input`*, not the weights themselves — and it's random: a higher-weight index is more likely to be drawn, but low-weight indices can still come up.

### Example 1 — basic usage
```python
probs = torch.tensor([0.1, 0.1, 0.8])
torch.multinomial(probs, num_samples=1)
```
```
tensor([2])
```
Index `2` is returned about 80% of the time across many runs, since it has 80% of the probability mass — but not always, that's the point of sampling rather than `argmax`.

### Example 2 — batched sampling
```python
probs = torch.tensor([[0.9, 0.1], [0.1, 0.9]])   # 2 independent distributions
torch.multinomial(probs, num_samples=1)
```
```
tensor([[0],
        [1]])
```
One independent draw per row — this is exactly the batch shape used during generation, `(B, 1)`.

### In this notebook
```python
probs = F.softmax(logits, dim=-1) # (B, C)
idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
```
`probs` holds one probability distribution over all `vocab_size` characters, per sequence in the batch. Sampling (rather than always taking the highest-probability character with `argmax`) is what makes `generate()` produce different text on every run, instead of the same deterministic output every time.

### Summary
| Call | Behavior |
|---|---|
| `torch.multinomial(probs, 1)` | draw 1 random index per row, weighted by `probs` |
| vs. `probs.argmax(-1)` | always the single most likely index — deterministic, no variety |

# Understanding `torch.cat()`

`torch.cat()` joins a list of tensors end-to-end along an *existing* dimension (unlike `torch.stack`, which adds a *new* dimension). All tensors must already agree in shape except along the dimension being joined.

### Syntax
```python
torch.cat(tensors, dim=0)
```
**Parameters:**
- `tensors`: a sequence of tensors, matching in every dimension except `dim`.
- `dim`: which existing axis to concatenate along.

### Example 1 — joining along the last axis
```python
a = torch.tensor([[1, 2]])   # shape (1, 2)
b = torch.tensor([[3]])      # shape (1, 1)
torch.cat((a, b), dim=1)
```
```
tensor([[1, 2, 3]])
```
Shape `(1, 3)` — the non-joined dimension (dim 0, size 1) had to match; the joined dimension's sizes just add (`2 + 1 = 3`).

### Example 2 — growing a sequence one token at a time
```python
seq = torch.tensor([[7]])          # shape (1, 1): one sequence, one token so far
new_token = torch.tensor([[42]])   # shape (1, 1): the next token
torch.cat((seq, new_token), dim=1)
```
```
tensor([[ 7, 42]])
```
Shape grows from `(1,1)` to `(1,2)` — exactly the pattern used to grow generated text one token at a time.

### In this notebook
```python
idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
```
Inside `generate`'s loop, `idx` holds every token generated *so far*, shape `(B, T)`. `idx_next` is the one freshly-sampled token per sequence, shape `(B, 1)`. Concatenating along `dim=1` (the time axis) appends it to the end of every sequence in the batch, growing `T` by exactly 1 each iteration — after `max_new_tokens` iterations, `idx` has grown from `(B, T)` to `(B, T + max_new_tokens)`.

### Summary
| Call | Result |
|---|---|
| `torch.cat((a, b), dim=1)` where `a` is `(B,T)`, `b` is `(B,1)` | `(B, T+1)` |
| used `max_new_tokens` times in a loop | sequence grows by 1 token per iteration |

### Exercise 8 — The toy BigramLanguageModel

Implement a bigram language model: an `nn.Embedding(vocab_size, vocab_size)` table where row `i` directly holds the logits for "what comes after token `i`". In `forward`, compute the loss with `F.cross_entropy` (remember it wants `(N, C)` logits and `(N,)` targets, so flatten `B` and `T` together). In `generate`, repeatedly: run the model, keep only the last position's logits, softmax them into probabilities, sample one token with `torch.multinomial`, and append it.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = ...  # nn.Embedding mapping each token id to vocab_size logits

    def forward(self, idx, targets=None):
        logits = ...  # (B,T,vocab_size) -- look up each token's row

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = ...    # flatten to (B*T, C) for cross_entropy
            targets = ...   # flatten to (B*T,)
            loss = ...      # F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = ...     # (B, C) -- keep only the last time step
            probs = ...      # softmax over the vocab dimension
            idx_next = ...   # sample one token id per batch row
            idx = ...        # append idx_next to idx along the time dimension
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

In [ ]:
%%ipytest -qq

def test_forward_with_targets():
    logits, loss = m(xb, yb)
    assert logits.shape == (batch_size * block_size, vocab_size)
    assert loss is not None and loss.item() > 0

def test_forward_without_targets():
    logits, loss = m(xb, None)
    assert loss is None
    assert logits.shape == (batch_size, block_size, vocab_size)

def test_generate_shape_and_range():
    out = m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=10)
    assert out.shape == (1, 11)
    assert out.dtype == torch.long
    assert (out >= 0).all() and (out < vocab_size).all()

**Why "bigram"?** `nn.Embedding(vocab_size, vocab_size)` is literally a `vocab_size × vocab_size` lookup table: row `i` holds the (unnormalized) logits for "what comes after token `i`". So this model predicts the next character using *only* the current character — no context at all, hence "bigram" (a 2-gram: previous token → next token). It's the simplest possible language model, useful here purely as scaffolding to get the training loop and generation loop working before adding any real context-mixing.

**Loss.** `logits` has shape `(B,T,vocab_size)` — for every position, an unnormalized score per possible next character. `F.cross_entropy` applies `softmax` then negative-log-likelihood internally:

$$\mathcal{L} = -\frac{1}{N}\sum_i \log \frac{e^{\text{logit}_{i,\,y_i}}}{\sum_j e^{\text{logit}_{i,j}}}$$

It requires a 2D `(N, C)` input and a 1D `(N,)` target, hence the `.view(B*T, C)` / `.view(B*T)` reshape — batch and time get flattened together because cross-entropy only cares about "which of the `C` classes is correct at each of the `N` positions," not which batch/time index that position came from. A uniform-random model over 65 classes gives loss `-log(1/65) ≈ 4.17` — compare that to the `4.88` printed above (untrained, in the same ballpark) and the `1.66` the full model reaches after training later on.

**`generate`.** This is autoregressive sampling — the same loop every LLM inference server runs: (1) run the model forward, (2) keep only the logits at the *last* position (`logits[:, -1, :]`), since that's the only "next token" prediction needed right now, (3) `softmax` to turn logits into a probability distribution, (4) `torch.multinomial` to sample one token from that distribution (stochastic, not `argmax` — so output varies run to run), (5) append the sampled token and repeat. Starting from `torch.zeros((1,1))` means "start generation from token id 0" (which happens to decode to `\n`).

# Understanding the PyTorch training step (`AdamW`, `zero_grad`, `backward`, `step`)

Every model in this notebook is trained with the same four-part pattern. Each part has a distinct job.

### The pattern
```python
optimizer = torch.optim.AdamW(model.parameters(), lr=...)   # once, before training
...
optimizer.zero_grad(set_to_none=True)   # every step
loss.backward()
optimizer.step()
```

**`torch.optim.AdamW(params, lr)`** — creates an optimizer object that knows about every trainable tensor in `params` (via `model.parameters()`), and will update them using the AdamW algorithm (Adam with decoupled weight decay — an adaptive, per-parameter learning-rate scheme, the standard choice for Transformers).

**`optimizer.zero_grad(set_to_none=True)`** — PyTorch *accumulates* gradients into `.grad` by default, so this must be called before every new `backward()` to clear out the previous step's gradients. `set_to_none=True` frees the old gradient tensors instead of overwriting them with zeros — a minor speed/memory optimization.

**`loss.backward()`** — runs backpropagation: walks the computation graph that produced `loss` and populates `.grad` on every parameter tensor that contributed to it, with `d(loss)/d(parameter)`.

**`optimizer.step()`** — uses those `.grad` values to actually update every parameter, according to the AdamW update rule and the learning rate.

### Example — a minimal loop
```python
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(100):
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
```
Each of the 100 iterations: compute a fresh loss on a fresh batch, clear old gradients, compute new gradients, nudge every parameter slightly to reduce that loss.

### In this notebook
The identical pattern appears twice — once for the toy bigram model (100 steps, `lr=1e-3`) and once for the full model (5000 steps, same `lr`, interleaved with periodic `estimate_loss()` calls to track train/val loss without updating parameters — note the `@torch.no_grad()` there, since evaluation shouldn't build a graph or accumulate gradients at all).

### Summary
| Call | Job |
|---|---|
| `AdamW(params, lr)` | create the optimizer, tell it what to update |
| `zero_grad()` | clear stale gradients |
| `backward()` | compute new gradients via autodiff |
| `step()` | apply one update using those gradients |

### Exercise 9 — Train the toy model

Build an `AdamW` optimizer over `m`'s parameters, then implement the standard four-part training step inside the loop: clear old gradients, backpropagate, and apply the update.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
optimizer = ...  # torch.optim.AdamW over m.parameters(), lr=1e-3

batch_size = 32
initial_weight = m.token_embedding_table.weight.clone().detach()  # snapshot, used by the tests below
losses = []
for steps in range(100):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    ...  # TODO: clear old gradients (set_to_none=True)
    ...  # TODO: backpropagate
    ...  # TODO: apply the optimizer update
    losses.append(loss.item())

print(loss.item())

In [ ]:
%%ipytest -qq

def test_gradients_and_params_updated():
    assert m.token_embedding_table.weight.grad is not None, "backward() doesn't seem to have run"
    assert not torch.equal(initial_weight, m.token_embedding_table.weight), "parameters should change after training"

def test_loss_decreases_on_average():
    avg_first = sum(losses[:10]) / 10
    avg_last = sum(losses[-10:]) / 10
    assert avg_last < avg_first, f"loss should decrease on average: first10={avg_first:.3f} last10={avg_last:.3f}"


The four-line training step is the same in essentially every PyTorch model you'll write: **forward** (compute `loss`), **`zero_grad`** (clear stale gradients left from the previous step — otherwise they'd silently accumulate), **`backward`** (populate `.grad` on every parameter via autodiff), **`step`** (have the optimizer nudge parameters using those gradients). `set_to_none=True` is a minor speed/memory optimization — it frees gradient tensors instead of overwriting them with zeros.

In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))

Still gibberish — expected. A bigram model's ceiling is low no matter how long you train it: each character is predicted from exactly one character of context, so it can learn plausible letter-*pairs* (e.g. "q" tends to be followed by "u") but nothing about words, grammar, or long-range structure. Fixing this requires the model to actually use its context window — which is exactly what attention provides, starting in the next section.

## The mathematical trick in self-attention

# Understanding `torch.tril()`

`torch.tril()` zeroes out everything **above** the main diagonal of a matrix, keeping the lower-triangular part (including the diagonal) unchanged. It's the standard way to build a causal mask.

### Syntax
```python
torch.tril(input, diagonal=0)
```
**Parameters:**
- `input`: a matrix (or batch of matrices).
- `diagonal` (optional): which diagonal to cut along. `0` (default) keeps the main diagonal; negative/positive values shift the cutoff.

### Example 1 — basic usage
```python
torch.tril(torch.ones(4, 4))
```
```
tensor([[1., 0., 0., 0.],
        [1., 1., 0., 0.],
        [1., 1., 1., 0.],
        [1., 1., 1., 1.]])
```
Row `i` has exactly `i+1` ones (positions `0..i`), zero elsewhere — row `i` "can see" columns `0` through `i`.

### Example 2 — as a boolean mask
```python
mask = torch.tril(torch.ones(3, 3)) == 0
mask
```
```
tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])
```
`True` marks exactly the "future" positions relative to each row — this is what gets fed to `masked_fill`.

### In this notebook
```python
tril = torch.tril(torch.ones(T, T))
...
self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
```
A `(T, T)` matrix where entry `(i, j)` is `1` if `j <= i` (position `j` is at or before position `i`) and `0` otherwise. Every self-attention `Head` uses this shape to decide, for every query position `i`, which key positions `j` it's even allowed to consider — `j <= i` only, which is exactly what makes the model autoregressive (never able to peek at the future).

### Summary
| Call | Meaning |
|---|---|
| `torch.tril(torch.ones(T,T))` | 1s where `j<=i`, 0s where `j>i` — "causal" allowed-positions template |
| `... == 0` | boolean mask marking exactly the disallowed (future) positions |

# Understanding `@` (matrix multiplication) as a weighted average

The `@` operator between two matrices does the usual linear-algebra matrix product — but it's worth internalizing one specific reading of it: **if one matrix's rows sum to 1, `@` computes a weighted average.**

### The general shape rule
```python
# (T, T) @ (T, C) -> (T, C)
# (B, T, T) @ (B, T, C) -> (B, T, C)   (batched; broadcasts over B)
```
Row `i` of the output = `sum_j( weights[i, j] * values[j] )` — a weighted combination of all rows of the second matrix, using row `i` of the first matrix as the weights.

### Example 1 — uniform averaging
```python
weights = torch.tril(torch.ones(3, 3))
weights = weights / weights.sum(1, keepdim=True)   # normalize rows to sum to 1
values = torch.tensor([[2., 7.], [6., 4.], [6., 5.]])
weights @ values
```
```
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
```
Row 0 of `weights` is `[1, 0, 0]`, so row 0 of the output is just `values[0]` unchanged. Row 2 is `[1/3, 1/3, 1/3]`, so row 2 of the output is the plain average of all three rows of `values`.

### Example 2 — non-uniform (data-dependent) weights
If `weights` isn't a fixed triangular matrix but instead comes from `softmax(Q @ K.T)` (as in real attention), the exact same `weights @ values` operation now computes a *learned*, content-dependent weighted average instead of a plain running mean — same matrix-multiply mechanics, different (trained) weights.

### In this notebook
This exact pattern appears three times, each version more general than the last: `wei @ x` with uniform triangular weights (the "trick"), then `wei @ v` inside real self-attention, where `wei = softmax(q @ k.T / sqrt(head_size))` are learned, data-dependent attention weights, and `v` (not the raw input `x`) is what actually gets aggregated.

### Summary
| Expression | What it computes |
|---|---|
| `(row-stochastic matrix) @ (data)` | a weighted average of the data's rows |
| `wei @ x` with uniform triangular `wei` | causal running mean |
| `wei @ v` with `wei = softmax(q@k.T)` | self-attention output |

### Exercise 10 — The averaging trick (toy matmul)

Build a row-normalized lower-triangular matrix `a` (so each row is a uniform average over itself and everything before it), then use `a @ b` to compute those weighted averages.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
torch.manual_seed(42)
a = ...  # torch.tril(torch.ones(3,3)), then normalize each row to sum to 1
b = torch.randint(0, 10, (3, 2)).float()
c = ...  # a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

In [ ]:
%%ipytest -qq

def test_a_is_uniform_causal_average():
    expected_a = torch.tensor([[1., 0, 0], [.5, .5, 0], [1/3, 1/3, 1/3]])
    assert torch.allclose(a, expected_a)

def test_c_is_weighted_average_of_b():
    assert torch.allclose(c[0], b[0])
    assert torch.allclose(c[1], (b[0] + b[1]) / 2)
    assert torch.allclose(c[2], b.mean(0))

This is the mathematical core the rest of self-attention builds on, so it's worth sitting with. `a` is lower-triangular and each row sums to 1 (a *row-stochastic* matrix), so `a @ b` computes, for row `i`, a weighted average of rows `0..i` of `b` — using only *past and current* rows, never future ones. Concretely, here every past row gets equal weight (`1/(i+1)`), i.e. a running mean. The punchline we're driving toward: if instead of uniform weights we make the weights **data-dependent** (computed from the vectors themselves rather than fixed constants), this exact "triangular weighted average" mechanism *is* causal self-attention.

### Exercise 11 — The bag-of-words loop (xbow)

For every batch element and time step `t`, average all vectors from position `0` to `t` (inclusive) — a plain, explicit double loop.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = ...       # x[b, :t+1] -- everything up to and including position t
        xbow[b, t] = ...  # mean of xprev over the time axis

In [ ]:
%%ipytest -qq

def test_xbow_shape():
    assert xbow.shape == (B, T, C)

def test_xbow_first_and_last_position():
    assert torch.allclose(xbow[0, 0], x[0, 0]), "position 0 has nothing before it -- should equal itself"
    assert torch.allclose(xbow[:, -1, :], x.mean(1)), "the last position should average the whole sequence"

def test_xbow_second_position():
    assert torch.allclose(xbow[0, 1], x[0, :2].mean(0))

Same result, `wei @ x` instead of nested loops: `wei` is exactly the `a` matrix from the earlier toy example (lower-triangular, row-normalized), broadcast over the batch dimension — `(T,T) @ (B,T,C) → (B,T,C)`. This is purely a vectorization win right now (`wei` is still fixed/uniform), but notice the *shape* of the computation (`weights @ values`) is already in the exact form attention needs.

### Exercise 12 — Vectorizing with matmul (version 2)

Reproduce `xbow` with a single matrix multiply instead of the double loop: build the same row-normalized triangular matrix, then apply it with `@`.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
wei = ...    # torch.tril(torch.ones(T,T)), row-normalized to sum to 1
xbow2 = ...  # wei @ x
torch.allclose(xbow, xbow2)

In [ ]:
%%ipytest -qq

def test_xbow2_matches_xbow():
    assert torch.allclose(xbow, xbow2, atol=1e-6)

# Understanding `tensor.masked_fill()`

`masked_fill(mask, value)` returns a copy of a tensor where every position where `mask` is `True` gets replaced by `value` — everywhere else is left untouched. It's how a boolean condition gets turned into actual numeric values.

### Syntax
```python
tensor.masked_fill(mask, value)
```
**Parameters:**
- `mask`: a boolean tensor, same shape as (or broadcastable to) `tensor` — `True` marks positions to overwrite.
- `value`: the number to write into every `True` position.

### Example 1 — basic usage
```python
x = torch.tensor([1., 2., 3., 4.])
mask = torch.tensor([False, True, False, True])
x.masked_fill(mask, 0.0)
```
```
tensor([1., 0., 3., 0.])
```

### Example 2 — masking with `-inf` ahead of softmax
```python
wei = torch.zeros(3, 3)
tril = torch.tril(torch.ones(3, 3))
wei.masked_fill(tril == 0, float('-inf'))
```
```
tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])
```
Every "future" position (where `tril == 0`, i.e. above the diagonal) becomes `-inf`; everything else is unchanged.

### In this notebook
```python
wei = wei.masked_fill(tril == 0, float('-inf'))
```
This is the causal mask in action: `wei` holds raw attention scores for every `(query, key)` pair; wherever the key position is in the future relative to the query (`tril == 0`), the score gets overwritten with `-inf`. The next line, `F.softmax(wei, dim=-1)`, then turns every `-inf` into exactly `0` probability — so information genuinely cannot flow backward from the future.

### Summary
| Call | Effect |
|---|---|
| `x.masked_fill(mask, 0.0)` | zero out masked positions |
| `wei.masked_fill(tril==0, float('-inf'))` | force masked positions to vanish after the next `softmax` |

### Exercise 13 — Softmax + masked_fill (version 3)

Same result a third way: start from all-zero affinities, mask out the future positions with `-inf`, then `softmax` to turn that into the uniform causal weights.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = ...  # mask future positions (tril == 0) with float('-inf')
wei = ...  # softmax over the last dimension
xbow3 = ...  # wei @ x
torch.allclose(xbow, xbow3)

In [ ]:
%%ipytest -qq

def test_xbow3_matches_xbow():
    assert torch.allclose(xbow, xbow3, atol=1e-6)

def test_wei_rows_sum_to_one():
    assert torch.allclose(wei.sum(-1), torch.ones(T))

Reformulating the fixed averaging matrix via `masked_fill(-inf)` + `softmax` looks like more work for the same answer, but it's the version that generalizes: `softmax` of all-zeros-except-masked gives uniform weights over the unmasked entries (same result as before), but if the *pre-softmax* values (`wei`, before masking) aren't uniform zeros and are instead computed from the data — that's the entire jump to real attention, made in the very next cell. The mask itself (`tril == 0 → -inf`) is what makes this **causal**: position `t` can only attend to positions `≤ t`, which is required for autoregressive training (the model must never get to peek at future tokens it's supposed to predict).

# Understanding `nn.Linear()`

`nn.Linear` is a fully-connected (dense) layer: it learns a weight matrix (and optionally a bias vector) that maps every input vector to an output vector of a chosen size, via `output = input @ W.T + b`.

### Syntax
```python
nn.Linear(in_features, out_features, bias=True)
```
**Parameters:**
- `in_features`: size of each input vector.
- `out_features`: size of each output vector.
- `bias` (optional): whether to add a learned bias vector. Defaults to `True`.

### Example 1 — basic usage
```python
layer = nn.Linear(4, 2)
x = torch.randn(1, 4)   # 1 example, 4 features
layer(x).shape
```
```
torch.Size([1, 2])
```

### Example 2 — applies only to the last dimension
```python
x = torch.randn(8, 32, 4)   # any leading dims
layer(x).shape
```
```
torch.Size([8, 32, 2])
```
`nn.Linear` only ever touches the *last* dimension — everything before it is treated as "batch" and left alone. This is why the same `nn.Linear(C, head_size)` can be applied to a whole `(B, T, C)` tensor at once, transforming every token's vector independently.

### In this notebook
```python
self.key = nn.Linear(n_embd, head_size, bias=False)
self.query = nn.Linear(n_embd, head_size, bias=False)
self.value = nn.Linear(n_embd, head_size, bias=False)
```
Three separate, independently-learned linear layers, each projecting every token's `n_embd`-dim vector down to a `head_size`-dim vector — one becomes that token's key, one its query, one its value. `bias=False` is a common simplification for attention projections. `self.proj = nn.Linear(n_embd, n_embd)` (in `MultiHeadAttention`) and every layer inside `FeedFoward` are the same building block, just with different in/out sizes.

### Summary
| Call | Maps |
|---|---|
| `nn.Linear(n_embd, head_size, bias=False)` | `(...,n_embd) -> (...,head_size)`, no bias |
| `nn.Linear(n_embd, 4*n_embd)` | expand the feed-forward's hidden layer |

# Understanding `tensor.transpose(dim0, dim1)`

`.transpose()` swaps two dimensions of a tensor, without changing any of the underlying values — it just changes which axis is "rows" and which is "columns" (or more generally, which two axes are swapped).

### Syntax
```python
tensor.transpose(dim0, dim1)
```
**Parameters:**
- `dim0`, `dim1`: the two dimensions to swap. Negative indices count from the end (`-1` = last dimension, `-2` = second-to-last).

### Example 1 — basic 2D case
```python
x = torch.tensor([[1, 2, 3], [4, 5, 6]])   # shape (2, 3)
x.transpose(0, 1)
```
```
tensor([[1, 4],
        [2, 5],
        [3, 6]])
```
Shape becomes `(3, 2)` — rows and columns swapped.

### Example 2 — the last two axes of a batched tensor
```python
k = torch.randn(4, 8, 16)     # (B, T, head_size)
k.transpose(-2, -1).shape      # torch.Size([4, 16, 8])
```
`-2` and `-1` refer to `T` and `head_size` here — the batch dimension `B` (axis 0) is left alone, only the last two axes swap.

### In this notebook
```python
wei = q @ k.transpose(-2, -1) # (B, T, C) @ (B, C, T) -> (B, T, T)
```
`q` is `(B, T, head_size)` and `k` is also `(B, T, head_size)` — you can't matrix-multiply two tensors both ending in `head_size`. Transposing `k`'s last two axes turns it into `(B, head_size, T)`, so `q @ k.transpose(-2,-1)` becomes `(B,T,head_size) @ (B,head_size,T) -> (B,T,T)` — a valid matmul that produces, for every pair of positions `(i,j)`, the dot product `q_i · k_j`.

### Summary
| Shape before | `.transpose(-2,-1)` | Shape after |
|---|---|---|
| `(B, T, head_size)` | swap last two axes | `(B, head_size, T)` |
| enables | `q @ k.transpose(-2,-1)` | `(B,T,T)` pairwise affinity matrix |

### Exercise 14 — Real self-attention (single head, raw ops)

Now make the weights data-dependent: project `x` into `key`/`query`/`value`, compute `q @ k^T` (scores), causal-mask it, softmax it, then use it to weight-average `value`.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = ...    # key(x)
q = ...    # query(x)
wei = ...  # q @ k.transpose(-2,-1)  -- (B,T,T) pairwise affinities

tril = torch.tril(torch.ones(T, T))
wei = ...  # mask future positions with -inf
wei = ...  # softmax over the last dimension

v = ...    # value(x)
out = ...  # wei @ v
out.shape

In [ ]:
%%ipytest -qq

def test_out_and_wei_shapes():
    assert out.shape == (B, T, head_size)
    assert wei.shape == (B, T, T)

def test_wei_rows_sum_to_one():
    assert torch.allclose(wei.sum(-1), torch.ones(B, T), atol=1e-5)

def test_wei_is_causal():
    upper = torch.triu(wei, diagonal=1)
    assert torch.allclose(upper, torch.zeros_like(upper)), "no position should attend to a future position"


**This cell is the entire idea of the Transformer.** Walk through it slowly:

- `key`, `query`, `value` are three independent linear projections of the same input `x`. Each token emits a **query** ("what am I looking for?"), a **key** ("what do I contain, for others to find?"), and a **value** ("what do I actually communicate, if attended to?"). Q and K live in the same space so they can be dot-producted; V doesn't have to.
- `wei = q @ k.transpose(-2,-1)` computes, for every pair of positions `(i,j)`, the dot product `q_i · k_j` — high when query `i` and key `j` "match." This is the (unnormalized, uncausaled) **attention score matrix**, shape `(T,T)`.
- Masking + softmax turns each row into a probability distribution over *allowed* (`j ≤ i`) positions — "how much should position `i` attend to each earlier position `j`."
- `out = wei @ v`: each output position becomes a weighted average of *value* vectors, weighted by how much attention that position pays to each earlier one.

The key upgrade over the previous section: **the weights are now data-dependent** — computed from `x` itself via learned projections — rather than fixed/uniform. That's the entire conceptual leap from "running mean" to "attention." Formally, for one head:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$$

where $M$ is the additive causal mask ($-\infty$ above the diagonal, $0$ on/below it) and $d_k$ = `head_size`. This cell doesn't yet include the $1/\sqrt{d_k}$ scaling — that's introduced (and motivated) a few cells below. Also note: this is "self"-attention because Q, K, and V all come from the same `x` — see the note further down on cross-attention for the alternative.

Row `t` of this matrix is literally "how much token `t` attends to each earlier token" (and itself), summing to 1. Note it's *already* non-uniform despite the model being randomly initialized and untrained — e.g. row 3 puts 58% of its weight on position 0. Once trained, these weights become interpretable-ish (a token often attends heavily to the subject of its clause, or a matching quote/bracket) — this is the basis of the attention-map visualizations you may have seen for real language models.

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

### Exercise 15 — Scaled attention

Confirm the variance argument for scaling: build unit-variance `q`/`k` and scale `q @ k^T` by `1/sqrt(head_size)`.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
wei = ...  # q @ k.transpose(-2,-1), scaled by head_size**-0.5

In [ ]:
%%ipytest -qq

def test_scaled_variance_is_roughly_one():
    v = wei.var().item()
    assert 0.3 < v < 5.0, f"scaled variance should be roughly O(1), got {v} (unscaled would be close to head_size={head_size})"


**Why scale by `1/sqrt(head_size)`?** If `q` and `k` have unit-variance entries (a reasonable initialization target), each dot product `q_i · k_j` sums `head_size` roughly-independent unit-variance terms, so `Var(q·k) ≈ head_size` — i.e. `wei` ends up with variance ≈ `head_size` (here ≈16), not 1, as confirmed by `wei.var()` above. Dividing by `sqrt(head_size)` restores unit variance regardless of `head_size`, which matters because...

...unscaled, large-magnitude logits make `softmax` saturate toward one-hot, as shown: multiplying by 8 turns a mild preference into an 80%-confident near-certainty. A saturated softmax has near-zero gradient almost everywhere (it's already near its extreme), which stalls learning — and gets worse the larger `head_size` is, since that's exactly what inflates the pre-scaling variance. Dividing by $\sqrt{d_k}$ keeps the softmax "diffuse" (soft) regardless of head size, which is why every Transformer implementation includes it — this full recipe is why it's called **scaled** dot-product attention.

### Checkpoint: self-attention derivation

1. Starting from the lower-triangular averaging matrix, what's the one change that turns "running mean of the past" into real attention?

<details><summary>Show answer</summary>

Making the averaging weights **data-dependent** instead of fixed/uniform. The lower-triangular matrix already gives a causal (only-look-at-the-past) weighted average — attention just replaces the fixed weights `1/(i+1)` with weights computed from the data itself, via `softmax(Q @ K^T)`.

</details>

2. Why must the causal mask be applied *before* the softmax, not after?

<details><summary>Show answer</summary>

Softmax needs to see `-inf` at the masked (future) positions so that, after exponentiating, those positions get *exactly* zero probability and every row still sums to 1. If you masked after softmax instead (e.g. zeroing out entries post-hoc), the remaining "allowed" probabilities would no longer sum to 1 — you'd need to renormalize, and you'd have thrown away the numerically clean `-inf → 0` guarantee.

</details>

3. What does each of Q, K, and V represent, and why don't K and V need to live in the same vector space as each other (only Q and K do)?

<details><summary>Show answer</summary>

Q = "what am I looking for," K = "what do I offer, to be matched against," V = "what do I actually send if attended to." Q and K must share a space because they're combined via a dot product (`Q @ K^T`) to produce a compatibility score. V never gets dot-producted with anything — it's only ever weighted-summed (`wei @ V`) — so its dimensionality is independent and can differ (in this notebook it doesn't, but it's allowed to).

</details>

4. Why does an unscaled `q @ k.T` have variance approximately equal to `head_size`, and what specifically goes wrong if you skip the `1/sqrt(head_size)` scaling?

<details><summary>Show answer</summary>

Each entry of `q @ k.T` is a dot product summing `head_size` terms; if `q` and `k` have unit-variance entries, those terms are roughly independent, so their variances add up to `≈ head_size`. Left unscaled, that inflated variance pushes softmax's input logits to large magnitudes, which makes softmax saturate toward one-hot — near-zero gradient almost everywhere, which stalls learning (and gets worse the larger `head_size` is).

</details>

Try answering purely from the derivation above (the cells building up from the triangular-matrix toy example) before checking your answers against it again.

### Exercise 16 — LayerNorm1d from scratch

Implement LayerNorm by hand: normalize each *row* (i.e. per training example, across its own features) to zero mean / unit variance, then apply the learned `gamma`/`beta` affine transform.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
class LayerNorm1d:

    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = ...  # mean over dim=1 (features), keepdim=True
        xvar = ...   # variance over dim=1, keepdim=True
        xhat = ...   # (x - xmean) / sqrt(xvar + eps)
        self.out = ...  # gamma * xhat + beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100)
x = module(x)
x.shape

In [ ]:
%%ipytest -qq

def test_layernorm_normalizes_per_row():
    torch.manual_seed(0)
    ln = LayerNorm1d(10)
    xin = torch.randn(5, 10) * 3 + 2
    yout = ln(xin)
    assert torch.allclose(yout.mean(1), torch.zeros(5), atol=1e-5)
    assert torch.allclose(yout.std(1, unbiased=True), torch.ones(5), atol=1e-2)

LayerNorm normalizes **each individual example's feature vector** to zero mean / unit variance (here: `x.mean(1)` — averaged over the *feature* dimension, per row), then applies a learned affine transform (`gamma`, `beta`) so the network can undo the normalization if that's actually better for a given layer. Contrast with BatchNorm, which normalizes each *feature* across the batch (`mean(0)` instead of `mean(1)`) — that makes BatchNorm's statistics depend on which other examples happen to share the batch (bad for variable/small batch sizes, and awkward for sequence models that may run batch size 1 at inference). LayerNorm has no cross-example dependency, which is why Transformers use it almost exclusively. (The comment `# used to be BatchNorm1d` is a nod to an earlier notebook in this series that used BatchNorm for MLPs — same code shape, just the normalization axis swapped.)

Formally, per row $x \in \mathbb{R}^d$:
$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2+\epsilon}}, \qquad y = \gamma \odot \hat{x} + \beta$$

This confirms the two normalization axes are different: the first check (`x[:,0]`, one feature across the whole batch) is *not* standardized — mean ≈0.15, std ≈0.88 — because that's not the axis LayerNorm normalizes; the second (`x[0,:]`, all features of one example) is essentially exactly standardized (mean ≈0, std =1), because that *is* the axis LayerNorm operates on.

This is a placeholder illustrating **cross-attention** and the encoder/decoder split from the original "Attention Is All You Need" paper: an *encoder* stack processes the source sentence (no causal mask — every token can see every other, since the whole sentence is known upfront) into key/value vectors; a *decoder* stack generates the target sentence autoregressively (causal mask, like our `Head` above), and at each decoder layer, queries come from the decoder's own sequence while keys/values come from the *encoder's* output — "cross" because Q comes from one sequence and K/V come from another. GPT is decoder-only (no encoder, no cross-attention): every `Head` in this notebook is self-attention because Q, K, and V all derive from the same `x`. This distinction only matters if you build a translation/seq2seq model instead of a plain autoregressive LM like the one we finish below.

### Full finished code, for reference

You may want to refer directly to the git repo instead though.

### How the pieces below fit together

This assembles everything above into an actual (tiny) GPT. Reading order for the classes, bottom-up:

- **`Head`** — exactly the self-attention head derived above, now parameterized by the real hyperparameters (`n_embd`, `block_size`, `dropout`) instead of the toy numbers used earlier. `register_buffer('tril', ...)` stores the causal mask as part of the module's state (moves with `.to(device)`, saved/loaded with the model) without making it a trainable parameter.
- **`MultiHeadAttention`** — runs `num_heads` independent `Head`s in parallel on the *same* input, concatenates their outputs, then projects back to `n_embd` with a linear layer. Why multiple heads instead of one bigger head? Each head can specialize (e.g. one tracks local syntax, another tracks long-range coreference) — concatenation lets the model combine several different attention patterns per layer instead of being forced into one. Note `num_heads * head_size == n_embd`, so the concatenation exactly refills the embedding dimension.
- **`FeedFoward`** — a per-position 2-layer MLP (`n_embd → 4·n_embd → n_embd` with a ReLU in between) applied identically and independently to every token. If attention is where tokens *exchange* information, the feed-forward is where each token *processes* what it just received — no cross-token interaction happens here at all.
- **`Block`** — one Transformer layer: `x = x + SelfAttention(LayerNorm(x))`, then `x = x + FeedForward(LayerNorm(x))`. Two things matter here: (1) the **residual/skip connections** (`x + ...`) — without them, gradients through many stacked layers vanish/explode; with them, each layer only needs to learn a small *update* to `x` rather than reproducing `x` from scratch, and the original `x` keeps a clean gradient path all the way back to the input. (2) **Pre-norm** (LayerNorm applied *before* each sublayer, not after, unlike the original 2017 paper) — this is the change GPT-2 onward adopted because it makes deep Transformers noticeably more stable to train.
- **`BigramLanguageModel`** (full version) — same class name as the toy model, now with real machinery: token embeddings **and** learned positional embeddings (`position_embedding_table`), summed together (this is how the model knows position at all — recall from the attention notes above that attention itself is permutation-invariant and has "no notion of space"), a stack of `n_layer` `Block`s, a final LayerNorm, then a linear `lm_head` projecting `n_embd → vocab_size` to get logits. `forward` and `generate` are structurally identical to the bigram version — that's the whole point of this exercise: *attention slots into the exact same training/generation scaffolding*, it just makes each token's representation actually depend on its context.
- **`estimate_loss`** — averages loss over `eval_iters` batches (instead of just one) for a less noisy read on train/val loss, and toggles `model.eval()`/`model.train()` around it — this matters once `dropout` (or BatchNorm) is in play, since those layers behave differently at eval time.

Compare the hyperparameters at the top of the next cell to GPT-3's, for a sense of scale: `n_embd`=64 here vs. 12288, `n_layer`=4 vs. 96, `n_head`=4 vs. 96, `block_size`=32 vs. 2048 — same architecture, roughly 800,000× fewer parameters (0.2M vs. ~175B). Everything you need conceptually to build GPT-3 is already in this cell; the rest is scale, systems engineering, and — for a chat-style model — instruction-tuning/RLHF on top.

# Understanding `nn.Dropout()`

`nn.Dropout(p)` randomly zeroes out a fraction `p` of its input's elements *during training only*, and rescales the rest to keep the overall magnitude roughly unchanged. It's a regularization technique — it prevents the network from relying too heavily on any single connection.

### Syntax
```python
nn.Dropout(p=0.5)
```
**Parameters:**
- `p`: probability that any given element gets zeroed out, each forward pass.

Behaves differently depending on `model.train()` vs `model.eval()` mode: active during training, a no-op (identity) during evaluation — exactly why `estimate_loss()` calls `model.eval()` before evaluating and `model.train()` afterward.

### Example
```python
layer = nn.Dropout(p=0.5)
layer.train()
layer(torch.ones(10))
```
```
tensor([2., 0., 2., 0., 2., 2., 0., 2., 0., 0.])
```
Roughly half the entries become `0`; survivors are scaled up by `1/(1-p) = 2` so the expected sum stays the same. `layer.eval()` would instead return the input completely unchanged.

### In this notebook
```python
self.dropout = nn.Dropout(dropout)   # inside Head, MultiHeadAttention, and FeedFoward
```
`dropout = 0.0` in this notebook's hyperparameters, so in practice these layers are a no-op here — they're included so the code matches a "real" GPT implementation, where `dropout` is typically `0.1`–`0.2` and meaningfully reduces overfitting on larger models / smaller datasets.

### Summary
| Mode | Effect of `nn.Dropout(p)` |
|---|---|
| `model.train()` | randomly zero a fraction `p` of elements, rescale survivors |
| `model.eval()` | identity — dropout fully disabled |

# Understanding `nn.ModuleList()`

`nn.ModuleList` is a Python-list-like container specifically for holding submodules. Unlike a plain Python list, wrapping submodules in `nn.ModuleList` makes PyTorch aware of them — their parameters show up in `.parameters()`, they move correctly with `.to(device)`, and they get saved/loaded with the rest of the model.

### Syntax
```python
nn.ModuleList(modules)
```
**Parameters:**
- `modules`: an iterable of `nn.Module` instances.

It supports indexing and iteration like a regular list, but deliberately does **not** define a `forward()` — you still write the loop yourself.

### Example
```python
heads = nn.ModuleList([nn.Linear(4, 2) for _ in range(3)])
x = torch.randn(1, 4)
[h(x) for h in heads]   # a plain Python list of 3 outputs, each shape (1,2)
```
If a plain Python `list` had been used instead of `nn.ModuleList`, the three `nn.Linear` layers' parameters would silently be invisible to `model.parameters()` — they'd never get trained.

### In this notebook
```python
self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
```
Creates `num_heads` independent `Head` instances (each with its own separately-learned key/query/value projections). `forward` then does `[h(x) for h in self.heads]` — running every head on the same input `x`, then concatenating their outputs. Because it's an `nn.ModuleList` and not a plain list, every head's `nn.Linear` weights correctly end up in `model.parameters()` and get updated by the optimizer.

### Summary
| Container | Parameters visible to `.parameters()` / optimizer? |
|---|---|
| plain Python `list` of modules | No — silently untrained |
| `nn.ModuleList` of modules | Yes |

# Understanding `nn.Sequential()`

`nn.Sequential` chains a list of modules together, automatically feeding each one's output into the next one's input. It's a shortcut for writing a `forward()` method that's just "apply these layers, one after another."

### Syntax
```python
nn.Sequential(module1, module2, ...)
```
Calling the resulting object on an input `x` is equivalent to `module2(module1(x))`, extended to however many modules were given.

### Example
```python
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2),
)
net(torch.randn(1, 4)).shape
```
```
torch.Size([1, 2])
```
No custom `forward` needed — `nn.Sequential` already knows to run the input through all three layers in order.

### In this notebook
Two different uses:
```python
self.net = nn.Sequential(
    nn.Linear(n_embd, 4 * n_embd),
    nn.ReLU(),
    nn.Linear(4 * n_embd, n_embd),
    nn.Dropout(dropout),
)
```
`FeedFoward`'s entire body is one `nn.Sequential` — expand, non-linearity, project back down, dropout — no explicit loop needed.
```python
self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
```
The `*[...]` unpacks a *list* of `n_layer` separately-constructed `Block` instances into `nn.Sequential`'s individual arguments — this is how the model stacks `n_layer` Transformer blocks and runs input through all of them with a single `self.blocks(x)` call.

### Summary
| Call | Behavior |
|---|---|
| `nn.Sequential(a, b, c)` | `c(b(a(x)))` |
| `nn.Sequential(*[Block(...) for _ in range(n_layer)])` | `n_layer` blocks chained, built from a list |

# Understanding `self.register_buffer()`

`register_buffer(name, tensor)` attaches a tensor to a module as persistent state that is **not** a trainable parameter. It behaves like `self.name = tensor`, except PyTorch also makes sure it moves along with `.to(device)` calls and gets included when the model is saved/loaded — but the optimizer will never update it, since it's not returned by `.parameters()`.

### Syntax
```python
self.register_buffer(name, tensor)
```
Call this inside a module's `__init__`, after `super().__init__()`. Afterward, access it as `self.<name>` like any other attribute.

### Example
```python
class Foo(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('mask', torch.tril(torch.ones(4, 4)))

f = Foo()
f.to('cpu')            # f.mask moves too, even though it's not a parameter
list(f.parameters())    # [] -- empty; mask is NOT trainable
```

### In this notebook
```python
self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
```
The causal mask is fixed math (it never needs to be learned — it's always "block everything after position `i`"), but it does need to exist on the right device (CPU or GPU, matching the rest of the model) and needs to be saved alongside the model's actual weights. `register_buffer` gives exactly that, without accidentally handing a giant constant matrix to the optimizer as if it were a trainable parameter.

### Summary
| Attribute type | Moves with `.to(device)`? | Trainable (in `.parameters()`)? |
|---|---|---|
| plain Python attribute (`self.x = tensor`) | No | No |
| `nn.Parameter` | Yes | Yes |
| buffer (`register_buffer`) | Yes | No |

# Understanding `@torch.no_grad()`

`torch.no_grad()` is a context manager (usable as a decorator, as in this notebook) that disables PyTorch's automatic gradient tracking for everything inside it. Operations still run normally and return normal tensors — they just don't build the computation graph that `.backward()` would otherwise need, which saves memory and some compute.

### Syntax
```python
@torch.no_grad()
def some_function():
    ...
```
or equivalently, as a `with` block:
```python
with torch.no_grad():
    ...
```

### Example
```python
x = torch.tensor([1.0], requires_grad=True)
with torch.no_grad():
    y = x * 2
y.requires_grad   # False -- no graph was recorded for this computation
```
Outside the `with` block, the same `x * 2` would set `y.requires_grad = True` and remember how to backprop through it.

### In this notebook
```python
@torch.no_grad()
def estimate_loss():
    ...
```
`estimate_loss()` only ever *reads* the loss to report it — it never calls `.backward()` or updates any parameters. Wrapping it in `no_grad()` tells PyTorch not to bother building a graph for any of the `eval_iters` forward passes inside it, which is both faster and uses meaningfully less memory (no need to keep every intermediate activation around for a backward pass that's never going to happen).

### Summary
| Context | Gradients tracked? | Typical use |
|---|---|---|
| normal (default) | Yes | training forward passes |
| inside `torch.no_grad()` | No | evaluation / inference-only code |

# Understanding `torch.arange()`

`torch.arange()` is PyTorch's version of Python's `range()` — it produces a 1D tensor of evenly-spaced integers (or floats), instead of a lazy Python range object.

### Syntax
```python
torch.arange(end)
torch.arange(start, end, step=1)
```

### Example
```python
torch.arange(5)
```
```
tensor([0, 1, 2, 3, 4])
```

### In this notebook
```python
pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
```
`torch.arange(T)` produces `[0, 1, 2, ..., T-1]` — exactly the sequence of position indices for a sequence of length `T`. Feeding that straight into `position_embedding_table` (an `nn.Embedding`) looks up the learned vector for "position 0," "position 1," ..., "position T-1," all in one call — giving a `(T, n_embd)` tensor that then gets broadcast-added onto every sequence in the batch's token embeddings.

### Summary
| Call | Result |
|---|---|
| `torch.arange(T)` | `[0, 1, ..., T-1]` |
| `position_embedding_table(torch.arange(T))` | that sequence's learned position vectors, `(T, n_embd)` |

In [ ]:
# hyperparameters (given)
batch_size = 16
block_size = 32
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
torch.manual_seed(1337)

### Exercise 17 — get_batch with device support

Same `get_batch` as before, now for the full model's `block_size`/`batch_size`, with one addition: move the batch onto `device` (GPU if available) before returning it.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = ...  # move x and y onto `device`
    return x, y

In [ ]:
%%ipytest -qq

def test_get_batch_shapes_and_device():
    x, y = get_batch('train')
    assert x.shape == (batch_size, block_size)
    assert str(x.device) == device
    assert str(y.device) == device

### Exercise 18 — estimate_loss

Average the loss over `eval_iters` batches for both `train` and `val`, so the reported number is far less noisy than a single batch's loss. Remember to switch the model to eval mode (disables dropout) and back to train mode afterward, and to run this under `@torch.no_grad()` since it's evaluation-only.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = ...   # model(X, Y)
            losses[k] = ...      # loss.item()
        out[split] = ...  # mean of losses
    model.train()
    return out

In [ ]:
%%ipytest -qq

def test_estimate_loss_structure():
    class _DummyModel(nn.Module):
        def forward(self, X, Y):
            return None, torch.tensor(1.23)
    global model
    model = _DummyModel()
    out = estimate_loss()
    assert set(out.keys()) == {'train', 'val'}
    assert abs(out['train'].item() - 1.23) < 1e-4
    assert abs(out['val'].item() - 1.23) < 1e-4

### Exercise 19 — Head (the real nn.Module)

The same single-head self-attention as before, now as a proper `nn.Module`: `key`/`query`/`value` linear projections, a causal mask stored via `register_buffer`, dropout on the attention weights, then the weighted aggregation of `value`.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = ...      # nn.Linear(n_embd, head_size, bias=False)
        self.query = ...    # nn.Linear(n_embd, head_size, bias=False)
        self.value = ...    # nn.Linear(n_embd, head_size, bias=False)
        ...  # self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = ...    # self.key(x)
        q = ...    # self.query(x)
        wei = ...  # q @ k.transpose(-2,-1) * C**-0.5
        wei = ...  # mask future positions using self.tril[:T, :T], fill with -inf
        wei = ...  # softmax over the last dimension
        wei = self.dropout(wei)
        v = ...    # self.value(x)
        out = ...  # wei @ v
        return out

In [ ]:
%%ipytest -qq

def test_head_output_shape():
    torch.manual_seed(0)
    head = Head(8)
    xin = torch.randn(1, 5, n_embd)
    out = head(xin)
    assert out.shape == (1, 5, 8)

def test_head_is_causal():
    torch.manual_seed(0)
    head = Head(8)
    x1 = torch.randn(1, 5, n_embd)
    out1 = head(x1)
    x2 = x1.clone()
    x2[0, -1, :] += 100.0   # perturb only the LAST token
    out2 = head(x2)
    assert torch.allclose(out1[0, :-1], out2[0, :-1], atol=1e-4), "earlier positions must not depend on a later token"


### Exercise 20 — MultiHeadAttention

Run several `Head`s in parallel on the same input, concatenate their outputs back into `n_embd` channels, then project and apply dropout.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = ...   # nn.ModuleList of num_heads Head(head_size) instances
        self.proj = ...    # nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = ...  # concatenate every head's output along the last dimension
        out = ...  # dropout(proj(out))
        return out

In [ ]:
%%ipytest -qq

def test_mha_output_shape_and_heads():
    mha = MultiHeadAttention(n_head, n_embd // n_head)
    xin = torch.randn(2, 10, n_embd)
    out = mha(xin)
    assert out.shape == (2, 10, n_embd)
    assert len(mha.heads) == n_head

### Exercise 21 — FeedFoward

A per-token 2-layer MLP: expand to `4*n_embd`, apply `ReLU`, project back down to `n_embd`, then dropout. No cross-token interaction here at all.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = ...  # nn.Sequential: Linear(n_embd,4*n_embd) -> ReLU -> Linear(4*n_embd,n_embd) -> Dropout(dropout)

    def forward(self, x):
        return ...  # self.net(x)

In [ ]:
%%ipytest -qq

def test_feedforward_shape():
    ff = FeedFoward(n_embd)
    xin = torch.randn(2, 10, n_embd)
    out = ff(xin)
    assert out.shape == xin.shape
    assert any(p.requires_grad for p in ff.parameters())

### Exercise 22 — Block

One Transformer block: self-attention and feed-forward, each wrapped in a **pre-norm residual connection** (`x = x + sublayer(LayerNorm(x))`).

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = ...    # MultiHeadAttention(n_head, head_size)
        self.ffwd = ...  # FeedFoward(n_embd)
        self.ln1 = ...   # nn.LayerNorm(n_embd)
        self.ln2 = ...   # nn.LayerNorm(n_embd)

    def forward(self, x):
        x = ...  # x + self.sa(self.ln1(x))
        x = ...  # x + self.ffwd(self.ln2(x))
        return x

In [ ]:
%%ipytest -qq

def test_block_shape_and_changes_input():
    block = Block(n_embd, n_head)
    xin = torch.randn(2, 10, n_embd)
    out = block(xin)
    assert out.shape == xin.shape
    assert not torch.allclose(out, xin)

### Exercise 23 — The full BigramLanguageModel

Assemble everything: token + position embeddings summed together, a stack of `n_layer` `Block`s, a final LayerNorm, and an `lm_head` projecting to `vocab_size` logits. `forward` and `generate` follow the exact same shape as the toy model — `generate` just needs one addition: crop the context to the last `block_size` tokens before every forward pass.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = ...     # nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = ...  # nn.Embedding(block_size, n_embd)
        self.blocks = ...   # nn.Sequential of n_layer Block(n_embd, n_head=n_head) instances
        self.ln_f = ...     # nn.LayerNorm(n_embd)
        self.lm_head = ...  # nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = ...  # self.token_embedding_table(idx)
        pos_emb = ...  # self.position_embedding_table(torch.arange(T, device=device))
        x = ...        # tok_emb + pos_emb
        x = ...        # self.blocks(x)
        x = ...        # self.ln_f(x)
        logits = ...   # self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = ...  # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters()) / 1e6, 'M parameters')

In [ ]:
%%ipytest -qq

def test_full_model_forward():
    torch.manual_seed(1337)
    model_ = BigramLanguageModel().to(device)
    xb_, yb_ = get_batch('train')
    logits, loss = model_(xb_, yb_)
    assert logits.shape == (batch_size * block_size, vocab_size)
    assert loss.item() > 0

def test_full_model_generate_shape():
    torch.manual_seed(1337)
    model_ = BigramLanguageModel().to(device)
    out = model_.generate(torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=5)
    assert out.shape == (1, 6)

def test_generate_crops_context_beyond_block_size():
    # if idx_cond doesn't crop to block_size, this raises an IndexError from position_embedding_table
    torch.manual_seed(1337)
    model_ = BigramLanguageModel().to(device)
    out = model_.generate(torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=block_size + 5)
    assert out.shape == (1, block_size + 6)

### Exercise 24 — Train the full model

The same four-part training step as the toy model, now for `max_iters` steps, periodically calling `estimate_loss()` to report train/val loss.

Replace every `...` with your implementation. Running the code cell before you do that will raise an error or produce an obviously wrong result — that's expected. Then run the test cell right after it: **all asserts must pass before moving on.**

In [ ]:
# TODO: implement this exercise
optimizer = ...  # torch.optim.AdamW(model.parameters(), lr=learning_rate)

losses_history = []
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    ...  # TODO: clear old gradients (set_to_none=True)
    ...  # TODO: backpropagate
    ...  # TODO: apply the optimizer update
    losses_history.append(loss.item())

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

In [ ]:
%%ipytest -qq

def test_loss_decreases_substantially():
    avg_first = sum(losses_history[:20]) / 20
    avg_last = sum(losses_history[-20:]) / 20
    assert avg_last < avg_first - 0.5, f"expected a substantial decrease over {len(losses_history)} steps: first20={avg_first:.3f} last20={avg_last:.3f}"

def test_final_loss_is_reasonable():
    assert losses_history[-1] < 3.0, f"final loss {losses_history[-1]:.3f} seems too high for a converged run"


### Tracing one token through the full model

Concrete shapes make this much less abstract. Take a single training example: `T = 32` tokens (`block_size`); batch size is irrelevant here, so drop `B` and just track one sequence.

1. **Input**: a sequence of 32 integer token ids, each in `[0, 65)`.
2. **Embedding**: `token_embedding_table` maps each id to a 64-dim vector → shape `(32, 64)`. `position_embedding_table` adds a learned 64-dim vector *per position* (0..31) → still `(32, 64)`, now position-aware.
3. **Inside one `Block`**: `LayerNorm` normalizes each of the 32 rows independently → `(32, 64)` unchanged in shape. `MultiHeadAttention` splits the 64 channels across `n_head=4` heads of `head_size=16` each: each head computes `(32,32)` attention weights and outputs `(32,16)`; the 4 heads' outputs are concatenated back to `(32,64)`, then linearly projected `(32,64) → (32,64)`. Added back onto the block's input (residual). Then `LayerNorm` again, then `FeedFoward` expands to `(32,256)` and back down to `(32,64)`, again added back via residual.
4. **Repeat step 3** `n_layer=4` times — same shape in, same shape out, at every block; only the *content* of the 64-dim vectors changes, accumulating more context each pass.
5. **Final LayerNorm**: `(32,64) → (32,64)`.
6. **`lm_head`**: one more linear layer, `(32,64) → (32,65)` — 65 logits (one per possible next character) at every one of the 32 positions.
7. **At generation time**, only the logits at the *last* position (`[-1,:]`, shape `(65,)`) get used — turned into probabilities via `softmax` and sampled from via `multinomial` to pick the next token.

Notice the embedding dimension (64) never changes size anywhere in the middle of the network — every block takes `(T, 64)` in and returns `(T, 64)` out. That's a deliberate design choice, and true of every real GPT: a fixed-width "residual stream" that each block reads from and writes back into, only ever changing shape at the very start (embedding) and very end (`lm_head`).

### Where does "0.209729 M parameters" come from?

Walking the parameter count for the hyperparameters used (`vocab_size=65`, `n_embd=64`, `block_size=32`, `n_head=4`, `n_layer=4`):

- **Token embedding**: `vocab_size × n_embd` = 65 × 64 = 4,160
- **Position embedding**: `block_size × n_embd` = 32 × 64 = 2,048
- **Per attention head**: 3 linear projections (`key`, `query`, `value`), each `n_embd × head_size` = 64 × 16 = 1,024, no bias → 3,072 per head. With `n_head=4` heads: 12,288 per block. Plus the output projection `n_embd × n_embd` = 64 × 64 = 4,096 (+64 bias) = 4,160 → **16,448 per block** for attention.
- **Feed-forward**: `n_embd → 4·n_embd` = 64×256 + 256 (bias) = 16,640, then `4·n_embd → n_embd` = 256×64 + 64 = 16,448 → **33,088 per block**.
- **2 LayerNorms per block**: `gamma` + `beta`, each length 64 → 256 per block (small, easy to undercount but negligible here).
- **Per block total**: 16,448 + 33,088 + 256 ≈ 49,792. Over `n_layer=4` blocks: **≈199,168**.
- **Final LayerNorm**: 128.
- **`lm_head`**: `n_embd × vocab_size` + bias = 64×65 + 65 = 4,225.

Total ≈ 4,160 + 2,048 + 199,168 + 128 + 4,225 ≈ **209,729** — matching the printed `0.209729 M parameters` exactly. The two embedding tables and the attention/feed-forward projections inside the 4 blocks dominate; LayerNorm parameters are essentially free. This is the same accounting real GPT parameter counts are built from — plug in GPT-3's hyperparameters (`n_embd`=12288, `n_layer`=96, `vocab_size`≈50k) to see where its ~175B comes from.

### Reading the result

Loss drops from ~4.4 (near random-guess level — recall `-ln(1/65) ≈ 4.17`) to 1.66 train / 1.82 val over 5000 steps, and the generated text — while nonsense as *English* — now has correctly-formed words, plausible character names, stage-direction-like structure, and roughly right punctuation, all learned purely from character sequences, with attention as the only mechanism for using context beyond the immediately preceding character.

**Where to go next, to build this yourself from scratch:**
1. Re-implement each class in this notebook from memory, in a plain `.py` file, without looking back — that's the real test of whether you can build it.
2. Swap the character tokenizer for a real one (`tiktoken`'s GPT-2 BPE) and rerun — `vocab_size` jumps to ~50k, so the embedding/`lm_head` tables get much bigger, but nothing else in the architecture changes.
3. Read Karpathy's [`nanoGPT`](https://github.com/karpathy/nanoGPT) — the same architecture, scaled up with production concerns (mixed precision, gradient accumulation, learning-rate schedules, multi-GPU training).
4. Read the two papers this notebook is a distillation of: *Attention Is All You Need* (Vaswani et al., 2017) for the architecture, and the GPT-2/GPT-3 papers for the decoder-only, pre-norm variant used here.

### Checkpoint: full architecture

1. In `Block.forward`, why is `x = x + self.sa(self.ln1(x))` written as an addition rather than `x = self.sa(self.ln1(x))`?

<details><summary>Show answer</summary>

That's the residual (skip) connection. Without it, gradients have to flow back through every stacked block's transformation, which vanishes or explodes as depth grows. With `x + f(x)`, each block only has to learn a small *update* to `x`, and the original `x` keeps a clean, direct gradient path all the way back to the input — this is what makes deep Transformers trainable at all.

</details>

2. What's the actual difference in what `MultiHeadAttention` computes versus a single `Head` with `head_size = n_embd`?

<details><summary>Show answer</summary>

A single big head produces exactly one attention pattern (one softmax distribution) per query position. `MultiHeadAttention` with `n_head` heads of `head_size = n_embd / n_head` runs `n_head` *independent* Q/K/V projections and *independent* softmaxes in parallel, then concatenates the results — so the model can attend to different things in different ways simultaneously (e.g. one head tracking local syntax, another tracking long-range references), rather than being forced into a single mixing pattern per layer.

</details>

3. Which part of the model is responsible for the network knowing *where* a token is in the sequence — and why is that needed at all, given how attention works?

<details><summary>Show answer</summary>

The `position_embedding_table`. Attention itself is permutation-invariant — it computes affinities from Q·K content, with no built-in sense of order — so without adding a positional signal, shuffling a sequence's tokens around wouldn't change any token's output at all. Summing token embeddings with position embeddings is what gives every position a distinguishable identity before attention ever runs.

</details>

4. If you doubled `n_layer` and `n_embd` but left `block_size` unchanged, what would get more expensive, and what would stay the same cost?

<details><summary>Show answer</summary>

More expensive: everything driven by embedding width — the attention projections and feed-forward layers scale roughly with `n_embd²` per layer, and doubling `n_layer` on top multiplies total compute/parameters further (rough ballpark: 2× layers × ~4× per-layer cost ≈ 8×). Unchanged: anything driven purely by sequence length — the attention matrix stays `(block_size, block_size)` per head, the position embedding table still has only `block_size` rows, and the model still can't see further back than `block_size` tokens of context.

</details>

If you can answer all four without re-reading the code, you understand the architecture well enough to reimplement it.

### FAQ / common pitfalls

- **"My loss isn't decreasing."** Check, in order: is `zero_grad` called every step (stale gradients accumulate silently otherwise)? Is the learning rate absurdly high/low? Is `targets` actually shifted by one position relative to `x` (an off-by-one here silently trains the model to predict the *current* token, which is trivial and won't generalize)?
- **"My attention output looks the same regardless of token order."** You probably forgot the positional embedding — without `position_embedding_table`, attention alone has no notion of order (it only sees which tokens are present and how they relate, not where), so shuffling a sequence's positions wouldn't change per-token outputs at all.
- **"Training works fine on short sequences but breaks/gives garbage past `block_size`."** Expected — the model (and its position embedding table) was never trained on positions beyond `block_size`. `generate()` handles this by cropping (`idx[:, -block_size:]`) before every forward pass; forgetting that crop is a common bug when adapting this code.
- **"Why does validation loss stay close to training loss the whole time, unlike a lot of ML models I've seen?"** At this scale (0.2M params, 1M characters of data) the model is heavily underfitting, not overfitting — there's far more data than capacity to memorize it. Overfitting becomes visible once you scale `n_embd`/`n_layer` up faster than the dataset.
- **"Do I need the causal mask if I'm not generating text autoregressively?"** No — if the whole input is known upfront (classification, a translation encoder, etc.) you want *bidirectional* attention, i.e. skip the `tril` masking entirely, per the encoder/decoder note above.
- **"Why `-inf` and not just a very negative number?"** `-inf` guarantees `softmax` produces *exactly* 0 for masked positions, with no risk of a tiny nonzero leaking through from a "large but finite" negative number, and no risk of numerical issues if the unmasked values happen to be large too.